In [1]:
import os

os.environ["HF_HOME"] = r"D:\RAG1\hf_cache"
os.environ["HF_HUB_CACHE"] = r"D:\RAG1\hf_cache\hub"

print(os.environ["HF_HOME"])
print(os.environ["HF_HUB_CACHE"])

D:\RAG1\hf_cache
D:\RAG1\hf_cache\hub


In [2]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader
dir_loader=DirectoryLoader(
    path='data/pdf',
    glob='**/*.pdf',
    loader_cls=PyMuPDFLoader
)
pdf_documents=dir_loader.load()


C:\Users\veerendra balaji\AppData\Local\Temp\ipykernel_21192\3063658437.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader
d:\RAG1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
pdf_documents

[Document(metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': '2025-09-14T09:15:41+00:00', 'source': 'data\\pdf\\BCS714A-module-1-textbook.pdf', 'file_path': 'data\\pdf\\BCS714A-module-1-textbook.pdf', 'total_pages': 34, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-09-14T09:15:41+00:00', 'trapped': '', 'modDate': 'D:20250914091541Z', 'creationDate': 'D:20250914091541Z', 'page': 0}, page_content='1\nBiological and Machine Vision\nThroughout this chapter and much of this book, the visual system of biological organ-\nisms is used as an analogy to bring deep learning to, um . . . life. In addition to conveying\na high-level understanding of what deep learning is, this analogy provides insight into\nhow deep learning approaches are so powerful and so broadly applicable.\nBiological Vision\nFive hundred fifty million years ago, in the prehistoric Cam

# chunking

In [4]:

from langchain_text_splitters import RecursiveCharacterTextSplitter
def split_documnets(documents,chunk_size=300,chunk_overlap=100):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""] 
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks=split_documnets(documents=pdf_documents)


Split 100 documents into 1269 chunks

Example chunk:
Content: 1
Biological and Machine Vision
Throughout this chapter and much of this book, the visual system of biological organ-
isms is used as an analogy to bring deep learning to, um . . . life. In addition t...
Metadata: {'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': '2025-09-14T09:15:41+00:00', 'source': 'data\\pdf\\BCS714A-module-1-textbook.pdf', 'file_path': 'data\\pdf\\BCS714A-module-1-textbook.pdf', 'total_pages': 34, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-09-14T09:15:41+00:00', 'trapped': '', 'modDate': 'D:20250914091541Z', 'creationDate': 'D:20250914091541Z', 'page': 0}


## Embedding and VectorStore

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
import os
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
class Embedding_Manager:
    def __init__(self,model_name:str="all-MiniLM-L6-V2"):
        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading Embedding_model:{self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model Loaded Sucessfull\n. Model Dimensions:{self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error Loading {self.model_name} : {e}")
            raise

    def generate_embedding(self,text:List[str])->np.ndarray:
        if not self.model:
            raise ValueError("Model Not Loaded")

        print(f"Generating Embedding for{len(text)} text..")
        embeddings=self.model.encode(text,show_progress_bar=True)
        print(f"Generated Embeddings with Shape{embeddings.shape}")
        return embeddings
        



In [15]:
class VectorStore:
    
    def __init__(self,collection_name:str="pdf_documents",persist_directory:str="data/vector_store"):
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)

            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,metadata={'description':"PDF document embedding for RAG"}

            )

            print(f"Vector Store Initialized . Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initialzing Vector Store;{e}")
            raise

    def add_embeddings(self,documents:List[any],embeddings:np.ndarray):
        if len(documents)!=len(embeddings):
            raise ValueError("Number Documents must match Number of Embeddings")

        print(f"Adding{len(documents)} to VectorStore")

        #prepare data for chromadb
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_lenght']=len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            try:
                self.collection.add(
                    ids=ids,
                    embeddings=embeddings_list,
                    metadatas=metadatas,
                    documents=documents_text
                )
                print(f"Successfully addded {len(documents)} to vectorestore")
                print(f"Documents in collection {self.collection.count()}")

            except Exception as e:
                print(f"Error in adding documents {e}")
                raise

vectorstore=VectorStore()
VectorStore



   



Vector Store Initialized . Collection: pdf_documents
Existing documents in collection: 1269


__main__.VectorStore

In [16]:
texts = [doc.page_content for doc in chunks]

embedding_manager = Embedding_Manager()

embeddings = embedding_manager.generate_embedding(text=texts)

vectorstore.add_embeddings(chunks, embeddings)

Loading Embedding_model:all-MiniLM-L6-V2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2096.55it/s]
C:\Users\veerendra balaji\AppData\Local\Temp\ipykernel_21192\3954868332.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model Loaded Sucessfull\n. Model Dimensions:{self.model.get_sentence_embedding_dimension()}")


Model Loaded Sucessfull
. Model Dimensions:384
Generating Embedding for1269 text..


Batches: 100%|██████████| 40/40 [00:23<00:00,  1.67it/s]


Generated Embeddings with Shape(1269, 384)
Adding1269 to VectorStore
Successfully addded 1269 to vectorestore
Documents in collection 1270
Successfully addded 1269 to vectorestore
Documents in collection 1271
Successfully addded 1269 to vectorestore
Documents in collection 1272
Successfully addded 1269 to vectorestore
Documents in collection 1273
Successfully addded 1269 to vectorestore
Documents in collection 1274
Successfully addded 1269 to vectorestore
Documents in collection 1275
Successfully addded 1269 to vectorestore
Documents in collection 1276
Successfully addded 1269 to vectorestore
Documents in collection 1277
Successfully addded 1269 to vectorestore
Documents in collection 1278
Successfully addded 1269 to vectorestore
Documents in collection 1279
Successfully addded 1269 to vectorestore
Documents in collection 1280
Successfully addded 1269 to vectorestore
Documents in collection 1281
Successfully addded 1269 to vectorestore
Documents in collection 1282
Successfully addded 1

In [21]:
class RagRetriever:
    def __init__(self,embedding_manager=Embedding_Manager,vectorstore=VectorStore):
        self.embedding_manager=embedding_manager
        self.vector_store=vectorstore

    def retrive(self,query:str,top_k:int=5,score_threshold:float =0.0)->List[Dict[str,Any]]:
        print(f"Retriveing Documents for Query : {query}")
        print(f"Top K : {top_k} and Threshold : {score_threshold}")

        query_embedding=self.embedding_manager.generate_embedding([query])[0]

        # search in vector Store
        try:
            results=self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrived_docs=[]

            if results['documents'] and results['documents'][0]:
                documents=results['documents'][0]
                metadatas=results['metadatas'][0]
                distances=results['distances'][0]
                ids=results['ids'][0]

                for i ,(doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                         
                         similarity_Score=1-distance

                         if similarity_Score>=score_threshold:
                              
                              retrived_docs.append({
                              'ids':doc_id,
                               'content':document,
                               'metadata':metadata,
                               'similarity_Score':similarity_Score,
                               'distance':distance,
                                'rank':i+1
                                            })
                              
                print(f"Retrived {len(retrived_docs)} documents(after filtering)")

            else:
                print("No document found")
        
            return retrived_docs
        
        except Exception as e:
            print(f"Error During retrival {e}")
            return[]
embedding_manager = Embedding_Manager()

rag_retriver = RagRetriever(
    embedding_manager,
    vectorstore
)

Loading Embedding_model:all-MiniLM-L6-V2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4643.16it/s]


Model Loaded Sucessfull
. Model Dimensions:384


C:\Users\veerendra balaji\AppData\Local\Temp\ipykernel_21192\3954868332.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model Loaded Sucessfull\n. Model Dimensions:{self.model.get_sentence_embedding_dimension()}")


In [22]:
rag_retriver

In [23]:
rag_retriver.retrive('Biological Vision')

Retriveing Documents for Query : Biological Vision
Top K : 5 and Threshold : 0.0
Generating Embedding for1 text..


Batches: 100%|██████████| 1/1 [00:00<00:00, 35.81it/s]

Generated Embeddings with Shape(1, 384)
Retrived 5 documents(after filtering)


[{'ids': 'doc_3a293f03_40',
  'content': 'Biological Vision\n7\nFigure 1.6\nA caricature of how consecutive layers of biological neurons represent\nvisual information in the brain of, for example, a cat or a human\nhead. Photons of light stimulate neurons located in the retina of each eye, and this raw',
  'metadata': {'file_path': 'data\\pdf\\BCS714A-module-1-textbook.pdf',
   'doc_index': 40,
   'format': 'PDF 1.7',
   'keywords': '',
   'moddate': '2025-09-14T09:15:41+00:00',
   'modDate': 'D:20250914091541Z',
   'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)',
   'trapped': '',
   'author': '',
   'subject': '',
   'total_pages': 34,
   'creationDate': 'D:20250914091541Z',
   'creationdate': '2025-09-14T09:15:41+00:00',
   'content_lenght': 256,
   'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)',
   'title': '',
   'page': 4,
   'source': 'data\\pdf\\BCS714A-module-1-textbook.pdf'},
  'similarity_Score': 0.478429913520813,
  'distance': 0.521570086479187,
  'ra

In [24]:
rag_retriver.retrive(' Neocognitron')

Retriveing Documents for Query :  Neocognitron
Top K : 5 and Threshold : 0.0
Generating Embedding for1 text..


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.39it/s]


Generated Embeddings with Shape(1, 384)
Retrived 5 documents(after filtering)


[{'ids': 'doc_e1949b1c_67',
  'content': 'potent property of the neocognitron and its deep learning descendants, we go\nthrough an interactive example at the end of this chapter that demonstrates it.8\nLeNet-5\nWhile the neocognitron was capable of, for example, identifying handwritten char-',
  'metadata': {'source': 'data\\pdf\\BCS714A-module-1-textbook.pdf',
   'modDate': 'D:20250914091541Z',
   'keywords': '',
   'page': 6,
   'trapped': '',
   'moddate': '2025-09-14T09:15:41+00:00',
   'creationdate': '2025-09-14T09:15:41+00:00',
   'file_path': 'data\\pdf\\BCS714A-module-1-textbook.pdf',
   'total_pages': 34,
   'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)',
   'doc_index': 67,
   'content_lenght': 247,
   'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)',
   'author': '',
   'creationDate': 'D:20250914091541Z',
   'subject': '',
   'format': 'PDF 1.7',
   'title': ''},
  'similarity_Score': 0.37656283378601074,
  'distance': 0.6234371662139893,
  'rank': 1},

In [35]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv('GROQ_API_KEY')
llm=ChatGroq(api_key=groq_api_key,model="openai/gpt-oss-20b",temperature=0.1,max_tokens=1024)

def rag_simple(query,retriver,llm,top_k:int=3):
    results=retriver.retrive(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to Answer Question"

    prompt=f"""Use the following Context to answer concisely
           context:
           {context}

            question:{query}
 

            Answer:"""
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [36]:
answer=rag_simple('What is Biological Vision',rag_retriver,llm)
print(answer)

Retriveing Documents for Query : What is Biological Vision
Top K : 3 and Threshold : 0.0
Generating Embedding for1 text..


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.56it/s]


Generated Embeddings with Shape(1, 384)
Retrived 3 documents(after filtering)
**Biological vision** is the natural process by which living organisms perceive light. Photons strike photoreceptor cells in the retina, converting light into electrical signals that travel through successive layers of neurons in the visual system (retina → optic nerve → visual cortex, etc.), ultimately forming the perception of visual scenes.


In [34]:
from groq import Groq

client = Groq()

models = client.models.list()

for model in models.data:
    print(model.id)

openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-86m
meta-llama/llama-prompt-guard-2-22m
groq/compound-mini
canopylabs/orpheus-arabic-saudi
allam-2-7b
whisper-large-v3-turbo
qwen/qwen3.6-27b
canopylabs/orpheus-v1-english
groq/compound
openai/gpt-oss-120b
qwen/qwen3.8-27b
whisper-large-v3
openai/gpt-oss-20b
